In [10]:
from sentence_transformers import SentenceTransformer

In [11]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
# lets embedd with simple example
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [13]:
# let us embedd a simimlar document

d = "You dont need to register, you are accepted. You can also just start learning and submitting homework without registering."
dv1 = model.encode(d)

In [14]:
# now let us compare the query using the dot product
v1.dot(dv1)

np.float32(0.31310531)

I have gotten 0.31, closer to 1(meaning similar in meaning)

In [15]:
# let us try unrelated query
q2 = "How to install Docker on Windows"
v2 = model.encode(q2)

In [16]:
# now let us compare the 2 vectors
v1.dot(v2)

np.float32(-0.14236721)

The first score for q1 vs d (0.31) is higher, so that query is more similar to the document about registration. The second score for q2 vs d sits near 0, because installing Docker has nothing to do with registration. A score near 0 means the two vectors are about as different as they can be.

That's the whole idea behind vector search: similar texts get similar vectors, and a dot product tells us how similar.

## Cosine similarity
---


The all-MiniLM-L6-v2 model outputs normalized vectors - vectors with unit length. When both vectors are normalized, the dot product equals cosine similarity. That's why the model documentation says it "uses cosine similarity."

Cosine similarity measures the angle between two vectors, ignoring their length:


- 1.0 = same direction (similar)
- 0.0 = perpendicular (unrelated)
- -1.0 = opposite direction (opposite meaning)
Formally, if theta is the angle between two vectors, cosine similarity is cos(theta):

- cos(0) = 1 - vectors point in the same direction
- cos(90) = 0 - vectors are perpendicular
- cos(180) = -1 - vectors point in opposite directions


Because our vectors are normalized, the dot product gives us cosine similarity directly. This is why we can use v1.dot(dv) to compare texts.

In practice, we rarely get cosine similarity below 0. The embedding model maps text to a region of the vector space where most vectors have positive components. There's no concept of "opposite meaning" that maps to a vector pointing the other way.



# Loading the data

In module 1 we created ingest.py for loading the FAQ data.

Download it into this project:


In [17]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py




7[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[https://raw.githubusercontent.]87[Files: 0  Bytes: 0  [0 B/s] Re]87Saving 'ingest.py.2'
87ingest.py.2          100% [=============================>]     340     --.-KB/s87HTTP response 200  [https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py]
87ingest.py.2          100% [=============================>]     340     --.-KB/s87[Files: 1  Bytes: 340  [76 B/s]]8

We use it here:

In [18]:
from ingest import load_faq_data

documents = load_faq_data()

In [19]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

Generating embeddings
Each document is a Python dictionary with a question and an answer. We embed both together. That way a query can match against the question text and the answer text in our index.

Build one text per document:

In [20]:
texts = []

for doc in documents:
    text = doc['question'] + " " + doc['answer']
    texts.append(text) 

Now we will generate the embeddings. we have about 1200 text  here. we wont hand it over to the model because it will take time, so we embed the text in batches.


we will use tqdm to monitor the progress

In [21]:
from tqdm .auto import tqdm

In [22]:
# now we chunk the dataset into batches of 50 and encode each 
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)


  0%|          | 0/28 [00:00<?, ?it/s]

1368

We end up with 1350 vectors. On a GPU this is fast. Most of us run on
Codespaces without a GPU, so it takes a bit, but it's a one-off.

We turn them into a 2-dimensional array (matrix) where

- rows are documents (vectors)
- columns are dimensions of the vectors

In [23]:
import numpy as np 
X = np.array(vectors)

In [24]:
X.shape

(1368, 384)

Calling X.shape returns (1350, 384) - number of documents vs number of dimensions.

# Vector Search (under the hood)

---

Scoring Documents

After embedding the documents when a new query comes in it will be embedded also then be compare to all document embeddings via a dot product, the n keep the similar ones who have a score closer to 1 (Our Cosine Similarity).

In [25]:
# All new queries are enbedded first
new_query = "Can i still join the course after the starting date?"
v_new_query = model.encode(new_query)

In [26]:
# now compute the dot product against all the documents
scores = X.dot(v_new_query)

This is matrix-vector multiplication. Each element `i` of scores is the cosine similarity between document `i(row i of X)` and `v_query`.

In [27]:
# but we can also compute that using for loop
score = [v_new_query.dot(X[i]) for i in range(len(X))]

But `X.dot(v_query)` is much faster. Numpy runs optimized C code instead of looping in Python, so the matrix version is hard to beat. The outcome is the same either way: one score per document.



#### **Best Match** 
---

the highest score is similar document

In [28]:
idx = np.argmax(scores)
idx, scores[idx]

# this returns the document with the highest score and its index

(np.int64(2), np.float32(0.75763226))

In [29]:
# let us see what documet it is 
documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

#### Top 5 results
---

usually we need more than a single best match, so we pull the top 5 results

In [30]:
list_m = [3, 5, 6, 9, 7 ,8, 9, 0, 2, 3, 2, 2]
lst_5 = list_m[-5:]
lst_5[::-1]

[2, 2, 3, 2, 0]

In [31]:
# np.argsort sort from lowest to highest, so the last 5 are the top results
top5 = np.argsort(scores)[-5:]

# or a much shorter one 
top5 = np.argsort(-scores)[:5] 

In [32]:
# it comes out from from smallest first so we sort
top5 = top5[:: -1]
top5

array([  7, 538, 925, 643,   2])

In [33]:
scores[top5]

array([0.561227  , 0.65017414, 0.7140769 , 0.75234026, 0.75763226],
      dtype=float32)

It looks cryptic the first time you see it. But it's a common way to turn a min-sort into a max-sort.

Let's read off the actual documents:

In [34]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.561227
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I follow the course after it finishes?', 'answer': 'Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.\n\nYou can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.', 'doc_id': '068529125b'}

0.65017414
{'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.', 'doc_id': '74eb249bbf'}

0.7140769
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can reg

This is vector search in its simplest form. We embed the query, compute dot products against all documents, and return the highest-scoring ones.

We return 5 and not the single best for a reason. The answer to a question can be spread across several documents. One holds part of it, another fills in the rest. Sometimes the top result isn't the right one but the second is. We send all 5 to the LLM and let it combine them.

The number 5 is a starting point, picked on gut feeling. Later, when we evaluate search quality, we can test whether 3 or 10 works better for our data.

Doing this by hand with numpy is fine for a small dataset. A larger one needs a library that also handles filtering and ranking. That's what we turn to next.

## **Vector Search with minsearch**
---

In the previous section we did vector search by hand with numpy. We embedded the query, computed dot products, and found the best matches. Writing the argsort and matrix code every time gets old, and it can't filter by course. So instead we'll use a library that wraps all of it.

We'll use minsearch, the small in-memory search library we already used in module 1 for text search. It has a VectorSearch class for vector search.

Both classes share the same API:

fit to index data
search to query
filter_dict in search to filter by keyword
It's the simplest way to get started with vector search.

In [35]:
# we already have tehe document and the vectors from earlier 
# now we index them
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

We pass the numpy array X with all embeddings and the list of documents as payload. The keyword_fields parameter works the same as in the text Index, so we can filter by course later.



### **Searching**

In [36]:
# lets search for a question

new_query2 = "I just discovered a cpurse, can i still join?"

v_new_query2 = model.encode(new_query2)

results = vindex.search(v_new_query2, num_results=5)

under the hood it does exactly what we did by hand, it computes the dot product between each vector after riltering and the query vector

In [37]:
# let s take a look at the top result
results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

It return the document about joining the course late:
```
{"id": "74eb249bbf",
 "course": "llm-zoomcamp",
 "section": "General Course-Related Questions",
 "question": "I just discovered the course. Can I still join?",
 "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions."}

#### **Filter by course**
---


Like the text index, we can filter by keyword fields. This matters for user experience. A student in LLM Zoom Camp doesn't care about answers from the data engineering course. So we narrow to their course first, then score only within it.

Pass a filter_dict:

In [38]:
results = vindex.search(
    v_new_query2,
    filter_dict={'course':'llm-zoomcamp'}, 
    num_results=5)

now that we can run vector seacrh let us use it in rag

# **RAG with Vector Search**
---

previously we have implemented RAG pipeline in 3 steps with keyword search, now let us switch to the vectorsearch and see how it will work

```python
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)
```

This implementation uses keyword search, since the RAG is modeular we will only switch the keword search with vector search while the rest stays the same


Using RAGBase
In module 1 we put all the RAG logic into a RAGBase helper class. It has search, build_prompt, and llm methods, so we only need to override search.

Download rag_helper.py (and ingest.py if you didn't get it earlier) into your project:



In [ ]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py



7[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[https://raw.githubusercontent.]87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87Saving 'rag_helper.py'
87rag_helper.py         98% [============================> ]     785     --.-KB/s87HTTP response 200  [https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py]
87rag_helper.py         98% [============================> ]     785     --.-KB/s87[Files: 1  Bytes: 794  [67 B/s]]8

7[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s]

In [41]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"), base_url="https//:openrouter.ai/api/v1")

In [42]:
# `now we download and index the data
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [43]:
# use the RAGBase class
from rag_helper import RAGBase

assistant = RAGBase(index=index, llm_client=openai_client)

In [ ]:
query = "I just found out about the program, can i still join?"
assistant.rag(query)